<div dir="rtl">

# جلسه ۸: معماری‌های مدرن CNN: از LeNet تا ResNet

**Modern CNN Architectures: from LeNet to ResNet**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/08.%20Modern_CNN_Architectures.ipynb)

**دوره یادگیری عمیق و شبکه‌های عصبی** — کیوان محمودی

### اهداف این جلسه

- مرور تاریخچه معماری‌های CNN و ایده کلیدی هر کدام
- درک چرایی فیلترهای ۳×۳ در VGG و ساخت بلوک‌های قابل تکرار
- آشنایی با مشکل تخریب (Degradation) در شبکه‌های خیلی عمیق
- پیاده‌سازی Residual Block و شبکه ResNet-20 از صفر
- مقایسه تجربی شبکه ساده (Plain) و ResNet با عمق یکسان روی CIFAR-10

> **نکته اجرا:** این نوتبوک روی GPU سریع‌تر اجرا می‌شود. در Colab از منوی `Runtime → Change runtime type` گزینه **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl">

## ۱. تاریخچه کوتاه: رقابت ImageNet

**ImageNet** (Deng et al., 2009) دیتاستی با بیش از ۱٫۲ میلیون تصویر آموزشی در ۱۰۰۰ کلاس است. رقابت سالانه ILSVRC روی این دیتاست (۲۰۱۰ تا ۲۰۱۷) موتور پیشرفت معماری‌های CNN بود:

| سال | مدل | ایده کلیدی | خطای Top-5 |
|---|---|---|---|
| 1998 | LeNet-5 | اولین CNN موفق: Conv → Pool → FC (برای ارقام دست‌نویس) | — |
| 2012 | AlexNet | CNN عمیق روی GPU، ReLU، Dropout، Data Augmentation | ۱۵٫۳٪ |
| 2014 | VGG | شبکه عمیق و یکنواخت فقط با کرنل‌های ۳×۳ | ۷٫۳٪ |
| 2014 | GoogLeNet | ماژول Inception: چند اندازه کرنل به‌صورت موازی، کانولوشن ۱×۱ | ۶٫۷٪ |
| 2015 | ResNet | اتصال‌های میان‌بر (Residual) و آموزش شبکه‌های ۱۵۲ لایه | ۳٫۶٪ |

پیروزی AlexNet در ۲۰۱۲ با فاصله‌ای چشمگیر از رتبه دوم (حدود ۲۶٪ خطا) آغاز موج یادگیری عمیق بود. در این جلسه ایده‌های VGG و ResNet را پیاده‌سازی می‌کنیم، چون این دو ایده تقریباً در همه معماری‌های امروزی (حتی Transformerها) حضور دارند.

</div>

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
torch.manual_seed(42)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

<div dir="rtl">

## ۲. LeNet-5: نقطه شروع

ساختار LeNet همان الگویی است که در جلسه قبل استفاده کردیم: کانولوشن و Pooling برای استخراج ویژگی و سپس لایه‌های Fully Connected. نسخه امروزی آن (با ReLU و MaxPool به‌جای Sigmoid و Average Pool اصلی):

</div>

In [ ]:
lenet = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(2),    # 28 -> 14
    nn.Conv2d(6, 16, kernel_size=5), nn.ReLU(), nn.MaxPool2d(2),              # 14 -> 10 -> 5
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
    nn.Linear(120, 84), nn.ReLU(),
    nn.Linear(84, 10),
)
print(f'LeNet-5 parameters: {count_params(lenet):,}')
print('output:', lenet(torch.randn(1, 1, 28, 28)).shape)

<div dir="rtl">

## ۳. VGG: قدرت کرنل‌های کوچک

ایده اصلی VGG (Simonyan & Zisserman, 2015): به‌جای کرنل‌های بزرگ (۱۱×۱۱ در AlexNet)، چند کرنل ۳×۳ را پشت سر هم قرار دهیم.

- **میدان دید یکسان:** دو لایه ۳×۳ میدان دید ۵×۵ و سه لایه ۳×۳ میدان دید ۷×۷ دارند.
- **پارامتر کمتر:** با $C$ کانال ورودی و خروجی، یک لایه ۵×۵ تعداد $25C^2$ وزن دارد، اما دو لایه ۳×۳ فقط $2\times 9C^2 = 18C^2$.
- **غیرخطی بودن بیشتر:** بین هر دو لایه یک ReLU قرار می‌گیرد؛ پس شبکه توابع پیچیده‌تری را می‌تواند بیان کند.

VGG معماری را **یکنواخت و قابل تکرار** کرد: بلوک‌هایی از چند Conv ۳×۳ و یک MaxPool، که در هر بلوک تعداد کانال‌ها دو برابر و ابعاد مکانی نصف می‌شود. این نوع طراحی بلوکی امروز استاندارد است:

</div>

In [ ]:
def vgg_block(n_convs, c_in, c_out):
    layers = []
    for i in range(n_convs):
        layers += [nn.Conv2d(c_in if i == 0 else c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU()]
    layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)


def make_vgg(config, n_classes=10):
    """config: list of (n_convs, out_channels) per block, e.g. VGG-11 style for 32x32 inputs."""
    blocks, c_in = [], 3
    for n_convs, c_out in config:
        blocks.append(vgg_block(n_convs, c_in, c_out))
        c_in = c_out
    return nn.Sequential(*blocks, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c_in, n_classes))


vgg11 = make_vgg([(1, 64), (1, 128), (2, 256), (2, 512), (2, 512)])
print(f'VGG-11 (CIFAR version) parameters: {count_params(vgg11):,}')

# parameter comparison: one 5x5 conv vs two stacked 3x3 convs (C = 64)
one_5x5 = nn.Conv2d(64, 64, 5, padding=2, bias=False)
two_3x3 = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.Conv2d(64, 64, 3, padding=1, bias=False))
print(f'one 5x5 conv: {count_params(one_5x5):,} weights | two 3x3 convs: {count_params(two_3x3):,} weights')

<div dir="rtl">

## ۴. مشکل تخریب (Degradation)

اگر عمق مفید است، چرا فقط لایه‌های بیشتری اضافه نکنیم؟ هی و همکاران (He et al., 2016) مشاهده کردند که با افزایش عمق یک شبکه ساده (Plain) از ۲۰ به ۵۶ لایه، **خطای آموزش** (نه فقط خطای آزمون) بیشتر می‌شود. این Overfitting نیست، چون مشکل روی داده آموزشی هم وجود دارد؛ یک **مشکل بهینه‌سازی** است.

استدلال جالب آن‌ها: یک شبکه عمیق‌تر حداقل باید به خوبی شبکه کم‌عمق‌تر باشد، چون لایه‌های اضافه می‌توانند تابع همانی (Identity) را یاد بگیرند. پس اینکه بدتر می‌شود یعنی **یادگرفتن تابع همانی با چند لایه غیرخطی برای SGD دشوار است**.

## ۵. ایده Residual Learning

راه‌حل ساده و درخشان: به‌جای اینکه از چند لایه بخواهیم تابع مطلوب $H(\mathbf{x})$ را مستقیم یاد بگیرند، از آن‌ها می‌خواهیم **باقیمانده (Residual)** $F(\mathbf{x}) = H(\mathbf{x}) - \mathbf{x}$ را یاد بگیرند و ورودی را از یک **اتصال میان‌بر (Shortcut / Skip Connection)** به خروجی اضافه می‌کنیم:

$$\mathbf{y} = F(\mathbf{x}) + \mathbf{x}$$

- اگر تابع همانی بهینه باشد، کافی است وزن‌های $F$ به سمت صفر بروند که بسیار ساده‌تر است.
- در Backpropagation، مشتق خروجی نسبت به ورودی برابر $\frac{\partial F}{\partial \mathbf{x}} + I$ است. جمله $I$ یعنی گرادیان همیشه یک **بزرگراه مستقیم** برای رسیدن به لایه‌های اول دارد و محو نمی‌شود.

وقتی ابعاد $F(\mathbf{x})$ و $\mathbf{x}$ متفاوت است (مثلاً وقتی با Stride ۲ ابعاد مکانی نصف و کانال‌ها دو برابر می‌شوند)، میان‌بر از یک **کانولوشن ۱×۱** با Stride ۲ عبور می‌کند (**Projection Shortcut**).

</div>

In [ ]:
class BasicBlock(nn.Module):
    """Residual block of ResNet-18/34 and the CIFAR ResNets: two 3x3 convs + shortcut."""

    def __init__(self, c_in, c_out, stride=1, residual=True):
        super().__init__()
        self.residual = residual
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False)   # BN makes the bias redundant
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:                                              # projection shortcut
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + self.shortcut(x)         # the residual connection
        return F.relu(out)


block = BasicBlock(16, 32, stride=2)
print('input  :', (4, 16, 32, 32))
print('output :', tuple(block(torch.randn(4, 16, 32, 32)).shape))

<div dir="rtl">

### ResNet برای CIFAR-10

مقاله ResNet برای CIFAR-10 خانواده‌ای از شبکه‌ها با عمق $6n+2$ معرفی کرد: یک لایه Conv ابتدایی، سه **مرحله (Stage)** که هر کدام $n$ بلوک دارند (با ۱۶، ۳۲ و ۶۴ کانال و ابعاد ۳۲، ۱۶ و ۸)، و در انتها Global Average Pooling و یک لایه Linear. با $n=3$ به **ResNet-20** با فقط حدود ۲۷۰ هزار پارامتر می‌رسیم.

با آرگومان `residual=False` همان معماری را **بدون** اتصال‌های میان‌بر می‌سازیم (Plain-20) تا مقایسه کاملاً منصفانه باشد:

</div>

In [ ]:
class CifarResNet(nn.Module):
    def __init__(self, n=3, residual=True, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        layers, c_in = [], 16
        for stage, c_out in enumerate([16, 32, 64]):
            for i in range(n):
                stride = 2 if (stage > 0 and i == 0) else 1         # downsample at the start of stages 2 and 3
                layers.append(BasicBlock(c_in, c_out, stride, residual))
                c_in = c_out
        self.layers = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, n_classes))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        return self.head(self.layers(self.stem(x)))


for n in (3, 5, 9):
    print(f'ResNet-{6 * n + 2}: {count_params(CifarResNet(n)):,} parameters')

<div dir="rtl">

### جریان گرادیان: Plain در برابر Residual

قبل از آموزش، نُرم گرادیان رسیده به لایه اول را در شبکه‌های Plain و Residual با عمق‌های ۲۰، ۵۶ و ۱۱۰ مقایسه می‌کنیم (مشابه آزمایش جلسه ۲، این بار روی CNN واقعی):

</div>

In [ ]:
def first_layer_grad_norm(model):
    model.train()
    x, y = torch.randn(32, 3, 32, 32), torch.randint(0, 10, (32,))
    loss = F.cross_entropy(model(x), y)
    model.zero_grad()
    loss.backward()
    return model.stem[0].weight.grad.norm().item()


for n in (3, 9, 18):
    torch.manual_seed(0)
    plain, res = CifarResNet(n, residual=False), CifarResNet(n, residual=True)
    print(f'depth {6 * n + 2:3d}: first-layer grad norm  plain = {first_layer_grad_norm(plain):.2e} | residual = {first_layer_grad_norm(res):.2e}')

<div dir="rtl">

شاید انتظار داشتید گرادیان شبکه ساده **محو** شود؛ اما در شبکه‌های ساده عمیق با BatchNorm، گرادیان لایه‌های اول با افزایش عمق به‌صورت نمایی **منفجر** می‌شود (Yang et al., 2019). در هر دو حالت نتیجه یکی است: سیگنال گرادیانی که به لایه‌های اول می‌رسد با عمق کنترل‌ناپذیر می‌شود و بهینه‌سازی ناپایدار است. در ResNet، به لطف مسیر همانی، نُرم گرادیان با عمق فقط به‌آرامی تغییر می‌کند.

## ۶. آزمایش: Plain-20 در برابر ResNet-20 روی CIFAR-10

دو شبکه را با تنظیمات مقاله (SGD با Momentum، نرخ یادگیری ۰٫۱، Weight Decay برابر $5\times10^{-4}$ و زمان‌بند کسینوسی) آموزش می‌دهیم. **این بخش را با GPU اجرا کنید**؛ روی CPU آموزش هر مدل چند ساعت طول می‌کشد.

</div>

In [ ]:
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
train_set = datasets.CIFAR10('data', train=True, download=True, transform=train_tf)
test_set = datasets.CIFAR10('data', train=False, download=True, transform=test_tf)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=512, num_workers=2, pin_memory=True)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, scheduler=None):
    """One pass over the training data. `scheduler` is stepped after every batch if given."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += criterion(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, optimizer, epochs, device, scheduler=None, verbose=True):
    """Train for `epochs` epochs and return the history of losses and accuracies."""
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler)
        va = evaluate(model, val_loader, criterion, device)
        for k, v in zip(history, (*tr, *va)):
            history[k].append(v)
        if verbose:
            print(f'epoch {epoch:2d} | train loss {tr[0]:.4f} acc {tr[1]:.4f} | val loss {va[0]:.4f} acc {va[1]:.4f}')
    return history

In [ ]:
EPOCHS = 30


def train_cifar(model, epochs=EPOCHS):
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.1, epochs=epochs, steps_per_epoch=len(train_loader))
    start = time.time()
    history = fit(model, train_loader, test_loader, optimizer, epochs, device, scheduler=scheduler, verbose=False)
    print(f'  {time.time() - start:.0f}s | final train acc {history["train_acc"][-1]:.4f} | test acc {history["val_acc"][-1]:.4f}')
    return history


results = {}
for name, residual in [('Plain-20', False), ('ResNet-20', True)]:
    print(name)
    torch.manual_seed(0)
    results[name] = train_cifar(CifarResNet(n=3, residual=residual))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in results.items():
    axes[0].plot(h['train_loss'], label=name)
    axes[1].plot(h['val_acc'], label=name)
axes[0].set_title('training loss'); axes[1].set_title('test accuracy')
for ax in axes:
    ax.set_xlabel('epoch'); ax.legend()
plt.show()

<div dir="rtl">

در عمق ۲۰ هر دو شبکه آموزش می‌بینند (به لطف BatchNorm و مقداردهی He)، اما ResNet Loss آموزش پایین‌تر و دقت بهتری دارد. فاصله این دو با افزایش عمق به‌شدت بیشتر می‌شود: در تمرین ۳ همین آزمایش را با ۵۶ لایه تکرار کنید.

## ۷. ResNetهای ImageNet و استفاده از torchvision

برای ImageNet، ResNet با یک لایه ۷×۷ و Max Pool شروع می‌شود و چهار مرحله دارد. در ResNet-50 و عمیق‌تر، به‌جای BasicBlock از **Bottleneck Block** استفاده می‌شود: کانولوشن ۱×۱ برای کاهش کانال‌ها، کانولوشن ۳×۳، و کانولوشن ۱×۱ برای افزایش دوباره کانال‌ها؛ این کار محاسبات را بسیار کم می‌کند.

همه این مدل‌ها در `torchvision.models` آماده‌اند (در جلسه بعد نسخه ازپیش‌آموزش‌دیده آن‌ها را استفاده می‌کنیم):

</div>

In [ ]:
from torchvision import models

for name, ctor in [('resnet18', models.resnet18), ('resnet50', models.resnet50), ('vgg16', models.vgg16),
                   ('mobilenet_v3_small', models.mobilenet_v3_small), ('efficientnet_b0', models.efficientnet_b0)]:
    m = ctor(weights=None)
    print(f'{name:20s} {count_params(m) / 1e6:6.1f} M parameters')

<div dir="rtl">

VGG-16 با ۱۳۸ میلیون پارامتر از ResNet-50 با حدود ۲۵ میلیون پارامتر ضعیف‌تر است؛ بیشتر پارامترهای VGG در لایه‌های Fully Connected انتهایی است که ResNet آن‌ها را با Global Average Pooling حذف کرده. معماری‌های بعدی مثل **MobileNet** (کانولوشن Depthwise Separable برای موبایل)، **EfficientNet** (مقیاس‌دهی متوازن عمق، پهنا و رزولوشن) و **ConvNeXt** (بازطراحی CNN با ایده‌های Transformer) همه روی پایه ResNet بنا شده‌اند.

## ۸. جمع‌بندی

- AlexNet (۲۰۱۲) آغازگر موج یادگیری عمیق بود: GPU، ReLU، Dropout و Augmentation.
- VGG: پشته‌ای از کرنل‌های ۳×۳ با میدان دید یکسان، پارامتر کمتر و غیرخطی بودن بیشتر؛ طراحی بلوکی یکنواخت.
- شبکه‌های ساده بسیار عمیق دچار **تخریب** می‌شوند: خطای آموزش با عمق زیاد می‌شود.
- ResNet با $\mathbf{y} = F(\mathbf{x}) + \mathbf{x}$ یادگیری تابع همانی را ساده و مسیری مستقیم برای گرادیان فراهم می‌کند.
- Projection Shortcut (Conv ۱×۱) وقتی ابعاد تغییر می‌کند؛ Bottleneck Block برای ResNetهای عمیق.
- اتصال‌های Residual امروز در تقریباً همه معماری‌ها، از جمله Transformer، حضور دارند.

## ۹. تمرین

۱. تعداد پارامترهای یک بلوک VGG با سه لایه ۳×۳ و ۲۵۶ کانال را با یک لایه ۷×۷ با همان کانال‌ها مقایسه کنید.

۲. یک **Bottleneck Block** (۱×۱ ← ۳×۳ ← ۱×۱ با ضریب گسترش ۴) پیاده‌سازی کنید و تعداد پارامتر آن را با دو BasicBlock هم‌خروجی مقایسه کنید.

۳. آزمایش بخش ۶ را با $n = 9$ (شبکه‌های ۵۶ لایه) تکرار کنید. فاصله Plain و ResNet چه تغییری می‌کند؟

۴. در `BasicBlock` جای ReLU آخر را عوض کنید و آن را قبل از جمع با میان‌بر قرار دهید. این تغییر چه اثری بر خاصیت «مسیر مستقیم گرادیان» دارد؟ (راهنمایی: مقاله Pre-activation ResNet از He et al., 2016.)

۵. Confusion Matrix مدل ResNet-20 را رسم کنید و آن را با مدل `CifarCNN` جلسه ۷ مقایسه کنید.

</div>